In [ ]:
import os
os.environ['SM_FRAMEWORK'] = 'tf.keras'  # Установка фреймворка для segmentation_models
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'  # Отключение проверки обновлений albumentations

# Установка scipy для чтения .mat файлов\n",
print("Installing scipy...")
try:
    import scipy.io
    print("scipy already installed.")
except ImportError:
    # !pip install --no-cache-dir scipy
    import scipy.io
    print("scipy installed successfully.")

# Убедитесь, что в настройках Kaggle включен интернет (Settings -> Internet: On)
print("Installing segmentation-models...")
try:
    import segmentation_models
    print("segmentation_models already installed.")
except ImportError:
    !pip install --no-cache-dir segmentation-models
    try:
        import segmentation_models as sm
        print("segmentation_models installed successfully.")
    except ImportError:
        raise ImportError("Failed to install segmentation_models. Please ensure internet is enabled in Kaggle settings and try again.")

import tensorflow as tf
import numpy as np
import cv2
from skimage.segmentation import find_boundaries
from sklearn.model_selection import train_test_split
import albumentations as A
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback, ReduceLROnPlateau
import matplotlib.pyplot as plt
import glob
import time
import scipy.io
from sklearn.metrics import accuracy_score
import segmentation_models as sm

# Настройка matplotlib для отображения в ячейках
%matplotlib inline
print("Matplotlib configured for inline display.")

# Настройка TensorFlow для использования GPU (если доступно)
try:
    physical_devices = tf.config.list_physical_devices('GPU')
    if len(physical_devices) > 0:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
            print(f"Configured memory growth for GPU: {device}")
    else:
        print("No GPU detected, running on CPU.")
except Exception as e:
    print(f"Error configuring GPU memory growth: {e}. Using default GPU configuration.")

In [ ]:
# Комбинированная функция потерь: BCE + Dice Loss с фиксированными весами классов
def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()(y_true, y_pred)
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    dice = (2. * intersection + 1.) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + 1.)
    weight_background = 1.0
    weight_cell = 2.0
    weights = tf.where(tf.keras.backend.greater(y_true_f, 0.5), weight_cell, weight_background)
    weighted_bce = tf.keras.backend.mean(weights * tf.keras.losses.binary_crossentropy(y_true_f, y_pred_f))
    return weighted_bce + (1 - dice)

# Вычисление весов классов
def compute_class_weights(mask_paths, num_samples=10):
    cell_pixels, total_pixels = 0, 0
    sample_indices = np.random.choice(len(mask_paths), min(num_samples, len(mask_paths)), replace=False)
    for idx in sample_indices:
        mask_data = scipy.io.loadmat(mask_paths[idx])
        mask = mask_data.get('inst_map', None)
        if mask is None:
            raise ValueError(f"No 'inst_map' key found in {mask_paths[idx]}. Check .mat file structure.")
        cell_mask = (mask > 0).astype(np.float32)
        cell_pixels += np.sum(cell_mask)
        total_pixels += cell_mask.size
    weight_cell = total_pixels / (2.0 * cell_pixels) if cell_pixels > 0 else 1.0
    weight_background = total_pixels / (2.0 * (total_pixels - cell_pixels)) if (total_pixels - cell_pixels) > 0 else 1.0
    return [float(weight_background), float(weight_cell)]

def load_paths(dataset_dirs, num_images=None):
    image_paths = []
    mask_paths = []
    
    for dataset_dir in dataset_dirs:
        images_dir = os.path.join(dataset_dir, 'Images')
        mask_dir = os.path.join(dataset_dir, 'Masks')
        img_ext = ['*.png']
        msk_ext = ['*.mat', '*.png']
        
        img_paths = []
        for ext in img_ext:
            img_paths.extend(sorted(glob.glob(os.path.join(images_dir, ext))))
        msk_paths = []
        for ext in msk_ext:
            msk_paths.extend(sorted(glob.glob(os.path.join(mask_dir, ext))))
        
        if len(img_paths) != len(msk_paths):
            print(f"Warning: Number of images ({len(img_paths)}) and masks ({len(msk_paths)}) mismatch in {dataset_dir}")
            continue
        
        image_paths.extend(img_paths)
        mask_paths.extend(msk_paths)
    
    if num_images is not None:
        image_paths = image_paths[:num_images]
        mask_paths = mask_paths[:num_images]
    
    print(f"Found {len(image_paths)} images and {len(mask_paths)} masks across all datasets")
    if len(image_paths) != len(mask_paths):
        raise ValueError(f"Number of images ({len(image_paths)}) does not match number of masks ({len(mask_paths)}).")
    
    print(f"Total images loaded: {len(image_paths)}")
    
    indices = np.arange(len(image_paths))
    train_val_indices, test_indices = train_test_split(indices, test_size=0.05, random_state=42)
    train_indices, val_indices = train_test_split(train_val_indices, test_size=0.15789473684210525, random_state=42)
    
    train_images = [image_paths[i] for i in train_indices]
    val_images = [image_paths[i] for i in val_indices]
    test_images = [image_paths[i] for i in test_indices]
    
    train_masks = [mask_paths[i] for i in train_indices]
    val_masks = [mask_paths[i] for i in val_indices]
    test_masks = [mask_paths[i] for i in test_indices]
    
    print(f"Train: {len(train_images)} images, Val: {len(val_images)} images, Test: {len(test_images)} images")
    return train_images, train_masks, val_images, val_masks, test_images, test_masks

class DataGenerator(Sequence):
    def __init__(self, image_paths, mask_paths, batch_size=8, image_size=512, augmentations=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.image_size = image_size
        self.augmentations = augmentations
        self.indices = np.arange(len(image_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, index):
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        images, targets = [], []
        for idx in batch_indices:
            # Загрузка изображения
            if self.image_paths[idx].endswith('.png'):
                image = cv2.imread(self.image_paths[idx])
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            elif self.image_paths[idx].endswith('.tif'):
                image = cv2.imread(self.image_paths[idx], cv2.IMREAD_UNCHANGED)
                if len(image.shape) == 2:
                    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB) if image.dtype == np.uint8 else cv2.cvtColor((image / np.max(image) * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
            
            # Загрузка маски с обработкой ошибок
            try:
                if self.mask_paths[idx].endswith('.png'):
                    mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
                    mask = (mask > 0).astype(np.float32)
                elif self.mask_paths[idx].endswith('.mat'):
                    mask_data = scipy.io.loadmat(self.mask_paths[idx])
                    mask = mask_data.get('inst_map', None)
                    if mask is None:
                        print(f"Warning: No 'inst_map' key found in {self.mask_paths[idx]}. Available keys: {list(mask_data.keys())}")
                        mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)  # Заглушка
                    else:
                        mask = (mask > 0).astype(np.float32)
            except ValueError as e:
                print(f"Error loading {self.mask_paths[idx]}: {e}. Skipping or using zero mask.")
                mask = np.zeros((self.image_size, self.image_size), dtype=np.float32)  # Заглушка при ошибке
            
            # Генерация boundary_mask
            boundary_mask = find_boundaries(mask, mode='inner').astype(np.float32)
            target = np.stack([mask, boundary_mask], axis=-1)
            
            # Ресайз, если нужно (хотя уже должно быть 512x512 из аугментаций)
            if image.shape[:2] != (self.image_size, self.image_size):
                image = cv2.resize(image, (self.image_size, self.image_size))
                target = cv2.resize(target, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
            
            if self.augmentations:
                augmented = self.augmentations(image=image, mask=target)
                image, target = augmented['image'], augmented['mask']
            
            images.append(image)
            targets.append(target)
        
        images = np.array(images) / 255.0
        targets = np.array(targets)
        return images, targets

    def on_epoch_end(self):
        np.random.shuffle(self.indices)

def plot_training_history(history):
    epochs = range(1, len(history.history['loss']) + 1)
    
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 3, 1)
    plt.plot(epochs, history.history['loss'], 'b-', label='Train Loss')
    plt.plot(epochs, history.history['val_loss'], 'r-', label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 3, 2)
    plt.plot(epochs, history.history['iou_score'], 'b-', label='Train IoU')
    plt.plot(epochs, history.history['val_iou_score'], 'r-', label='Val IoU')
    plt.title('Training and Validation IoU')
    plt.xlabel('Epoch')
    plt.ylabel('IoU Score')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 3, 3)
    plt.plot(epochs, history.history['precision'], 'b-', label='Train Precision')
    plt.plot(epochs, history.history['val_precision'], 'r-', label='Val Precision')
    plt.title('Training and Validation Precision')
    plt.xlabel('Epoch')
    plt.ylabel('Precision')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 3, 4)
    plt.plot(epochs, history.history['recall'], 'b-', label='Train Recall')
    plt.plot(epochs, history.history['val_recall'], 'r-', label='Val Recall')
    plt.title('Training and Validation Recall')
    plt.xlabel('Epoch')
    plt.ylabel('Recall')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 3, 5)
    plt.plot(epochs, history.history['f1_score'], 'b-', label='Train F1')
    plt.plot(epochs, history.history['val_f1_score'], 'r-', label='Val F1')
    plt.title('Training and Validation F1-Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 3, 6)
    plt.plot(epochs, history.history['accuracy'], 'b-', label='Train Accuracy')
    plt.plot(epochs, history.history['val_accuracy'], 'r-', label='Val Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    output_path = '/kaggle/working/training_history.png'
    plt.savefig(output_path)
    if os.path.exists(output_path):
        print(f"Training history saved to {output_path}")
    else:
        print(f"Failed to save training history to {output_path}")
    plt.show()
    plt.close()

def test_model(model_path):
    try:
        model = tf.keras.models.load_model(model_path, custom_objects={
            'iou_score': sm.metrics.IOUScore(threshold=0.5),
            'f1_score': sm.metrics.FScore(threshold=0.5),
            'precision': sm.metrics.Precision(threshold=0.5),
            'recall': sm.metrics.Recall(threshold=0.5),
            'bce_dice_loss': bce_dice_loss
        })
        print(f"Model loaded from {model_path}")
    except Exception as e:
        print(f"Error loading model: {e}")
        print("Model loading failed. Checking file existence and content...")
        if os.path.exists(model_path):
            print(f"File {model_path} exists. Size: {os.path.getsize(model_path)} bytes")
        else:
            print(f"File {model_path} does not exist!")
        return

    test_images_data = [cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB) for path in test_images[:2]]
    test_images_data = np.array(test_images_data) / 255.0
    predictions = model.predict(test_images_data, batch_size=8)
    
    iou_scores, f1_scores, precision_scores, recall_scores, accuracy_scores = [], [], [], [], []
    for i in range(2):
        true_mask_data = scipy.io.loadmat(test_masks[i])
        true_mask = true_mask_data.get('inst_map', None)
        if true_mask is None:
            raise ValueError(f"No 'inst_map' key found in {test_masks[i]}. Check .mat file structure.")
        true_mask = np.stack([(true_mask > 0).astype(np.float32), find_boundaries(true_mask > 0, mode='inner').astype(np.float32)], axis=-1)
        pred_mask = (predictions[i] > 0.5).astype(np.float32)
        
        iou = sm.metrics.IOUScore(threshold=0.5)(true_mask, pred_mask).numpy()
        f1 = sm.metrics.FScore(threshold=0.5)(true_mask, pred_mask).numpy()
        precision = sm.metrics.Precision(threshold=0.5)(true_mask, pred_mask).numpy()
        recall = sm.metrics.Recall(threshold=0.5)(true_mask, pred_mask).numpy()
        accuracy = tf.keras.metrics.BinaryAccuracy(threshold=0.5)(true_mask, pred_mask).numpy()
        
        iou_scores.append(iou)
        f1_scores.append(f1)
        precision_scores.append(precision)
        recall_scores.append(recall)
        accuracy_scores.append(accuracy)
    
    print(f"Average IoU: {np.mean(iou_scores):.4f}, F1-Score: {np.mean(f1_scores):.4f}, "
          f"Precision: {np.mean(precision_scores):.4f}, Recall: {np.mean(recall_scores):.4f}, "
          f"Accuracy: {np.mean(accuracy_scores):.4f}")
    
    # Визуализация изначальных и предсказанных масок для 2 случаев
    plt.figure(figsize=(15, 6))
    for i in range(2):
        plt.subplot(2, 4, i * 4 + 1)
        plt.imshow(test_images_data[i])
        plt.title(f'Image {i}')
        plt.axis('off')
        
        true_mask_data = scipy.io.loadmat(test_masks[i])
        true_mask = (true_mask_data.get('inst_map', None) > 0).astype(np.float32)
        plt.subplot(2, 4, i * 4 + 2)
        plt.imshow(true_mask, cmap='gray')
        plt.title(f'True Mask {i}')
        plt.axis('off')
        
        pred_mask = (predictions[i, :, :, 0] > 0.5).astype(np.float32)
        plt.subplot(2, 4, i * 4 + 3)
        plt.imshow(pred_mask, cmap='gray')
        plt.title(f'Predicted Mask {i}')
        plt.axis('off')
        
        plt.subplot(2, 4, i * 4 + 4)
        plt.imshow(test_images_data[i])
        plt.imshow(pred_mask, cmap='gray', alpha=0.5)
        plt.title(f'Overlay {i}')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Пользовательский коллбэк для ограничения времени
class TimeLimitCallback(Callback):
    def __init__(self, max_time_minutes):
        super(TimeLimitCallback, self).__init__()
        self.max_time_minutes = max_time_minutes
        self.start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        current_time = time.time()
        elapsed_minutes = (current_time - self.start_time) / 60
        if elapsed_minutes >= self.max_time_minutes:
            print(f"\nTraining stopped after {elapsed_minutes:.2f} minutes (reached {self.max_time_minutes} minutes limit).")
            self.model.stop_training = True


In [ ]:
train_augmentations = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30, p=0.5),
  #  A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
  #  A.GaussianBlur(blur_limit=(3, 7), p=0.3),
  #  A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.3),
])

val_augmentations = A.Compose([
    A.Resize(512, 512),
])

dataset_dirs = [
    '/kaggle/input/forarticle/DatasetFull'
]
try:
    train_images, train_masks, val_images, val_masks, test_images, test_masks = load_paths(dataset_dirs, num_images=200)
except ValueError as e:
    print(f"Error: {e}")
    exit()

print("Computing class weights...")
class_weights = compute_class_weights(train_masks)
print(f"Class weights (background, cell): {class_weights}")
weight_background, weight_cell = class_weights

train_generator = DataGenerator(train_images, train_masks, batch_size=8, augmentations=train_augmentations)
val_generator = DataGenerator(val_images, val_masks, batch_size=8, augmentations=val_augmentations)

try:
    model = sm.Unet(
        backbone_name='resnet34',
        encoder_weights=None,
        classes=2,
        activation='sigmoid',
        decoder_use_batchnorm=True,
        decoder_filters=(256, 128, 64, 32, 16),
        decoder_block_type='upsampling',
        encoder_freeze=False
    )
except Exception as e:
    print(f"Error creating model: {e}. Attempting to reset GPU configuration and retry.")
    tf.keras.backend.clear_session()
    model = sm.Unet(
        backbone_name='resnet34',
        encoder_weights=None,
        classes=2,
        activation='sigmoid',
        decoder_use_batchnorm=True,
        decoder_filters=(256, 128, 64, 32, 16),
        decoder_block_type='upsampling',
        encoder_freeze=False
    )

model.compile(
    optimizer='adam',
    loss=bce_dice_loss,
    metrics=[
        sm.metrics.IOUScore(threshold=0.5, name='iou_score'),
        sm.metrics.FScore(threshold=0.5, name='f1_score'),
        sm.metrics.Precision(threshold=0.5, name='precision'),
        sm.metrics.Recall(threshold=0.5, name='recall'),
        tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='accuracy')
    ]
)

callbacks = [
    EarlyStopping(patience=100, verbose=1),
    #TimeLimitCallback(max_time_minutes=240),
    ModelCheckpoint('/kaggle/working/best_model.h5', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
]


In [ ]:
start_time = time.time()
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks
)
end_time = time.time()
training_time = end_time - start_time
hours, rem = divmod(training_time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Training completed in {int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}")

In [ ]:
plot_training_history(history)

In [ ]:
model.save('/kaggle/working/final_model.h5')
print("Final model saved to /kaggle/working/final_model.h5")

In [ ]:
test_model('/kaggle/working/final_model.h5')